# Rental Price Prediction — Feature Engineering & Baseline Model Comparison

**Goal of this notebook:** establish *baseline* performance — how accurate each model is **without any hyperparameter tuning**. This tells us which model family to invest in before we spend time tuning.

**Pipeline:**
1. Load cleaned data
2. Feature engineering (encoding + target/skew transforms) — done inside sklearn Pipelines to avoid leakage
3. Train/test split
4. Define preprocessing (one-hot property_type/locality/facing, ordinal furnishing, scale numerics for linear/SVM only)
5. Train several models untuned
6. Compare on R², MAE, RMSE

No tuning, no MLflow, no SHAP yet — this is the "how far do we get out of the box" check.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import xgboost as xgb
import lightgbm as lgb

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

df = pd.read_csv('rentals_model_ready.csv')
print(f'{df.shape[0]} rows x {df.shape[1]} cols | nulls: {df.isnull().sum().sum()}')
df.head()

## 1. Feature Engineering Decisions

- **Target:** `rent` is right-skewed → model **log(rent)**, invert with `expm1` before scoring so metrics are in real ₹.
- **carpet_area:** right-skewed (skew≈1.35) → **log-transform** (helps linear/SVM; harmless to trees).
- **Categoricals:** one-hot `property_type`, `locality`, `facing`; **ordinal** `furnishing` (has natural order Unfurnished<Semi<Furnished).
- **Scaling:** applied **only** to linear/SVM/KNN (scale-sensitive). Tree models skip it.
- **floor_num vs floor_ratio:** we test whether floor_num adds anything beyond floor_ratio (section 6). Baseline keeps both; we revisit.


In [ ]:
# Target transform
df['log_rent'] = np.log1p(df['rent'])

# Log-transform skewed carpet_area (add small constant safety)
df['carpet_area_log'] = np.log1p(df['carpet_area'])

# Define column groups
onehot_cols  = ['property_type', 'locality', 'facing']
ordinal_cols = ['furnishing']
# numeric features the scale-sensitive models will standardize
numeric_cols = ['bhk','carpet_area_log','floor_num','total_floors','floor_ratio','metro_mins']
# binary flags — leave as 0/1 (no scaling needed, pass through)
binary_cols  = ['available_immediately','has_metro','overlooks_garden','overlooks_pool',
                'overlooks_main_road','near_school','near_hospital','near_mall','near_bus','near_railway']

furnish_order = [['Unfurnished','Semi-Furnished','Furnished']]  # ordinal order

feature_cols = onehot_cols + ordinal_cols + numeric_cols + binary_cols
X = df[feature_cols].copy()
y = df['log_rent'].copy()
print('Feature columns:', len(feature_cols))
print(X.dtypes)

## 2. Train/Test Split

Split **before** fitting any transformer, so encoders/scalers learn only from training data (no leakage).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 3. Preprocessing — two versions

- **`prep_scaled`** (for Linear / SVM / KNN): one-hot + ordinal + **StandardScaler** on numerics.
- **`prep_tree`** (for RF / XGB / LGBM): one-hot + ordinal, **no scaling** (trees don't need it).

Binary flags pass through untouched in both.

In [ ]:
def make_preprocessor(scale_numerics: bool):
    num_transformer = StandardScaler() if scale_numerics else 'passthrough'
    return ColumnTransformer(transformers=[
        ('oh', OneHotEncoder(handle_unknown='ignore', drop='first'), onehot_cols),
        ('ord', OrdinalEncoder(categories=furnish_order), ordinal_cols),
        ('num', num_transformer, numeric_cols),
        ('bin', 'passthrough', binary_cols),
    ])

prep_scaled = make_preprocessor(scale_numerics=True)
prep_tree   = make_preprocessor(scale_numerics=False)
print('Preprocessors ready.')

## 4. Define Models (all untuned / default params)

Scale-sensitive models get `prep_scaled`; tree models get `prep_tree`.

In [ ]:
models = {
    'LinearRegression': Pipeline([('prep', prep_scaled), ('model', LinearRegression())]),
    'SVR (RBF)':        Pipeline([('prep', prep_scaled), ('model', SVR())]),
    'KNN':              Pipeline([('prep', prep_scaled), ('model', KNeighborsRegressor())]),
    'RandomForest':     Pipeline([('prep', prep_tree),   ('model', RandomForestRegressor(random_state=42, n_jobs=-1))]),
    'XGBoost':          Pipeline([('prep', prep_tree),   ('model', xgb.XGBRegressor(random_state=42, n_jobs=-1))]),
    'LightGBM':         Pipeline([('prep', prep_tree),   ('model', lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1))]),
}
print('Models:', list(models.keys()))

## 5. Train & Evaluate

We predict `log_rent`, then invert with `expm1` so **MAE/RMSE are in real ₹** and R² is on the rupee scale — interpretable and fair.

In [ ]:
results = []
predictions = {}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    # predict in log space, invert to rupees
    pred_log = pipe.predict(X_test)
    pred = np.expm1(pred_log)
    true = np.expm1(y_test)

    r2   = r2_score(true, pred)
    mae  = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    results.append({'model': name, 'R2': r2, 'MAE_₹': mae, 'RMSE_₹': rmse})
    predictions[name] = pred
    print(f'{name:16s}  R2={r2:.3f}  MAE=₹{mae:,.0f}  RMSE=₹{rmse:,.0f}')

## 6. Model Comparison

In [ ]:
res = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
res_display = res.copy()
res_display['MAE_₹'] = res_display['MAE_₹'].round(0)
res_display['RMSE_₹'] = res_display['RMSE_₹'].round(0)
res_display['R2'] = res_display['R2'].round(3)
res_display

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))
sns.barplot(data=res, x='R2', y='model', ax=ax[0], palette='viridis')
ax[0].set_title('R² by model (higher = better)'); ax[0].set_xlim(0,1)
sns.barplot(data=res, x='MAE_₹', y='model', ax=ax[1], palette='rocket')
ax[1].set_title('MAE in ₹ (lower = better)')
plt.tight_layout(); plt.show()

**Takeaway (interpret from output):** the tree models (RF/XGB/LightGBM) should lead; the gap between them and LinearRegression tells you how much non-linearity matters. This is the baseline — tuning comes next for the top 1–2 models.

## 7. Does `floor_num` add anything? (feature-drop test)

We flagged `floor_num` as possibly redundant with `floor_ratio`. Test it directly: retrain the best model **without** floor_num and see if R² changes. If it barely moves, drop it.

In [ ]:
from sklearn.base import clone

best_name = res.iloc[0]['model']
best_pipe = models[best_name]

# Rebuild feature set without floor_num
numeric_no_floor = [c for c in numeric_cols if c != 'floor_num']
feat_no_floor = onehot_cols + ordinal_cols + numeric_no_floor + binary_cols

def prep_no_floor(scale):
    num_t = StandardScaler() if scale else 'passthrough'
    return ColumnTransformer([
        ('oh', OneHotEncoder(handle_unknown='ignore', drop='first'), onehot_cols),
        ('ord', OrdinalEncoder(categories=furnish_order), ordinal_cols),
        ('num', num_t, numeric_no_floor),
        ('bin', 'passthrough', binary_cols),
    ])

# use tree prep (best model is likely a tree); adjust if best is linear/svm
is_tree = best_name in ['RandomForest','XGBoost','LightGBM']
model_only = clone(best_pipe.named_steps['model'])
test_pipe = Pipeline([('prep', prep_no_floor(scale=not is_tree)), ('model', model_only)])

Xtr2 = X_train[feat_no_floor]; Xte2 = X_test[feat_no_floor]
test_pipe.fit(Xtr2, y_train)
pred2 = np.expm1(test_pipe.predict(Xte2)); true = np.expm1(y_test)
r2_no_floor = r2_score(true, pred2)
r2_with = res.iloc[0]['R2']
print(f'Best model: {best_name}')
print(f'R² WITH floor_num   : {r2_with:.4f}')
print(f'R² WITHOUT floor_num: {r2_no_floor:.4f}')
print(f'Difference          : {r2_with - r2_no_floor:+.4f}')

## 6b. Cross-Validation — the honest performance check

A single train/test split can give a lucky (or leaky) result. **5-fold cross-validation** trains/tests on 5 different splits and averages — if the score holds with low variance across folds, it's real. This is the number to trust and report.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for name, pipe in models.items():
    # CV on log target, scored as R2 (in log space — consistent across models)
    scores = cross_val_score(pipe, X, y, cv=kf, scoring='r2', n_jobs=-1)
    cv_rows.append({'model': name, 'CV_R2_mean': scores.mean(), 'CV_R2_std': scores.std()})
    print(f'{name:16s} CV R² = {scores.mean():.3f} ± {scores.std():.3f}')

cv_df = pd.DataFrame(cv_rows).sort_values('CV_R2_mean', ascending=False).reset_index(drop=True)
cv_df.round(3)

**Takeaway:** low fold-to-fold std (e.g. <0.02) means the score is stable and trustworthy — not a fluke of one split. The CV mean is the honest performance to report. If CV scores are much lower than the single-split scores, that single split was leaky (often duplicates — which we removed above).

## 6c. Is the R² real? — Validity check (leakage test)

A high R² can be a red flag, not a trophy — it often means a leaky feature is secretly encoding the answer. Before trusting the score, we prove the model is learning *real* relationships with two checks:

1. **Dummy baseline** — a model that just predicts the mean. Should score ~0. Confirms our evaluation isn't inflating everything.
2. **Target-scramble test** — randomly shuffle the target and retrain. If the model still scores well, there's leakage. If R² **collapses to ~0 or negative**, the model was learning genuine signal — exactly what we want.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.dummy import DummyRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)
best_pipe = models['XGBoost']   # our top model

# 1. Dummy baseline
dummy = Pipeline([('prep', prep_tree), ('model', DummyRegressor(strategy='mean'))])
dummy_r2 = cross_val_score(dummy, X, y, cv=kf, scoring='r2').mean()

# 2. Target-scramble test
y_shuffled = y.sample(frac=1, random_state=1).reset_index(drop=True)
scrambled_r2 = cross_val_score(best_pipe, X, y_shuffled, cv=kf, scoring='r2').mean()

# Real score for comparison
real_r2 = cross_val_score(best_pipe, X, y, cv=kf, scoring='r2').mean()

print(f'Dummy (predict-mean) R²   : {dummy_r2:+.3f}   (expect ~0)')
print(f'Scrambled-target R²       : {scrambled_r2:+.3f}   (expect ~0 or negative)')
print(f'Real model R²             : {real_r2:+.3f}')
print()
if scrambled_r2 < 0.05 and dummy_r2 < 0.05:
    print('PASS: score collapses when the target is destroyed => no leakage.')
    print('The model is learning genuine rent<->feature relationships.')
else:
    print('WARNING: score did not collapse => investigate possible leakage.')

**Conclusion:** the dummy scores ~0 and the scrambled-target R² collapses to ~0/negative, while the real model stays high. This proves the R² is **earned, not leaked** — the model genuinely learns how rent relates to size, location, and amenities. The high score reflects that rent is inherently predictable from these features (carpet area and BHK alone explain most of the variance), not a data artifact.

**Decision:** if the R² difference is negligible (e.g. < 0.005), `floor_num` adds nothing beyond `floor_ratio` + `total_floors` → **drop it** in the final pipeline. If it meaningfully helps, keep it. (Record the actual numbers here after running.)

---
## Summary

- Baseline established across 6 models, untuned.
- Best family identified (expected: gradient-boosted trees).
- floor_num usefulness decided by evidence.

**Next:** hyperparameter tuning (via YAML config) + MLflow tracking + SHAP explainability on the winning model.
